# 00 - Local PostgreSQL Setup

Creates LangGraph checkpoint tables + `scm_conversation_history`.

**Pre-req**: `docker-compose up postgres -d`

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

PG_USER     = os.getenv('PG_USER')
PG_PASSWORD = os.getenv('PG_PASSWORD')
PG_HOST     = os.getenv('PG_HOST', 'localhost')
PG_PORT     = os.getenv('PG_PORT', '5432')
PG_DATABASE = os.getenv('PG_DATABASE', 'postgres')

from urllib.parse import quote
DB_URI = (
    f'postgresql://{quote(PG_USER, safe="")}:{quote(PG_PASSWORD, safe="")}'
    f'@{PG_HOST}:{PG_PORT}/{PG_DATABASE}'
)
DB_KWARGS = dict(host=PG_HOST, port=int(PG_PORT), dbname=PG_DATABASE, user=PG_USER, password=PG_PASSWORD)

print(f'Target: {PG_HOST}:{PG_PORT}/{PG_DATABASE}')

Target: localhost:5432/postgres


## 1. Test connection

In [2]:
import psycopg

with psycopg.connect(DB_URI) as conn:
    print(conn.execute('SELECT version()').fetchone()[0])
print('Connection OK')

PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35225, 64-bit
Connection OK


## 2. PostgresSaver tables

In [3]:
from langgraph.checkpoint.postgres import PostgresSaver

with PostgresSaver.from_conn_string(DB_URI) as saver:
    saver.setup()

with psycopg.connect(DB_URI) as conn:
    tables = [r[0] for r in conn.execute(
        "SELECT table_name FROM information_schema.tables WHERE table_schema='public' ORDER BY 1"
    ).fetchall()]
    print(f'Tables: {tables}')
print('PostgresSaver ready')

Tables: ['checkpoint_blobs', 'checkpoint_migrations', 'checkpoint_writes', 'checkpoints', 'scm_conversation_history']
PostgresSaver ready


## 3. Conversation history table

In [4]:
import psycopg2

conn = psycopg2.connect(**DB_KWARGS)
try:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS scm_conversation_history (
                session_id  TEXT PRIMARY KEY,
                user_name   TEXT NOT NULL,
                title       TEXT,
                created_at  TIMESTAMPTZ DEFAULT NOW(),
                updated_at  TIMESTAMPTZ DEFAULT NOW(),
                turn_count  INTEGER DEFAULT 0,
                messages    TEXT
            )
        """)
        cur.execute('CREATE INDEX IF NOT EXISTS idx_scm_user ON scm_conversation_history(user_name)')
        cur.execute('CREATE INDEX IF NOT EXISTS idx_scm_upd  ON scm_conversation_history(updated_at DESC)')
    conn.commit()
    print('scm_conversation_history ready')
except Exception as e:
    conn.rollback(); raise
finally:
    conn.close()

scm_conversation_history ready


## 4. Smoke test

In [5]:
conn = psycopg2.connect(**DB_KWARGS)
try:
    with conn.cursor() as cur:
        cur.execute(
            'INSERT INTO scm_conversation_history (session_id, user_name, title) VALUES (%s,%s,%s) ON CONFLICT DO NOTHING',
            ('test-001', 'test@local.com', 'Smoke test')
        )
        cur.execute("SELECT session_id FROM scm_conversation_history WHERE user_name='test@local.com'")
        print(f'Read: {cur.fetchall()}')
        cur.execute("DELETE FROM scm_conversation_history WHERE user_name='test@local.com'")
    conn.commit()
    print('PostgreSQL fully operational')
finally:
    conn.close()

Read: [('test-001',)]
PostgreSQL fully operational
